# Notebook 20 — Super-Ensemble + Final Submission

**Goal:** Combine LGBM NB12 + MLP NB15 + LSTM NB18 + Hybrid GRU+FC NB21 via equal-weight rank average.

**ρ gate relaxation (confirmed after NB21):** Every neural model with solo AUC ≥ 0.905 clusters at
ρ ~0.91–0.92 vs MLP — a structural pattern, not architecture-specific. The ρ < 0.90 vs MLP gate is
impossible for any capable neural model. The relevant diversity is vs the GBM anchor (all neural models
ρ ~0.80 vs LGBM). Revised gate: **solo AUC ≥ 0.905 AND ρ < 0.85 vs LGBM**.

| Model | Solo CV AUC | Solo LB AUC | ρ vs MLP | ρ vs LGBM | NB20 status |
|-------|------------|------------|----------|-----------|------------|
| LGBM NB12 re-tuned | 0.9024 | 0.92840 | 0.791 | — | ✅ In (GBM anchor) |
| MLP NB15 | 0.9102 | — | — | 0.791 | ✅ In |
| LSTM NB18 | 0.9122 | — | 0.9126 | ~0.80 | ✅ In (ρ gate relaxed) |
| Hybrid GRU+FC NB21 | 0.9185 | 0.93618 | 0.9186 | 0.8023 | ✅ In (ρ gate relaxed) |
| ~~TFT NB19~~ | ~~0.8483~~ | — | 0.639 | 0.549 | ❌ AUC gate FAIL |

**CV→LB boost: +0.0177** (confirmed v006 Hybrid solo)
**Actual CV:** 0.9205 | **Actual LB:** 0.93463 (boost +0.0141) | **Output:** `submissions/submission_v007_super_ensemble.csv`

> **Result:** LB 0.93463 — WORSE than Hybrid solo (0.93618, −0.00155). Neural-on-neural ensemble failure
> confirmed: MLP+Hybrid ρ=0.9186 adds zero CV gain; LSTM+Hybrid ρ=0.9111 dilutes Hybrid on LB.
> Only LGBM is genuinely diverse (ρ=0.8023) but LGBM+Hybrid CV gain +0.0006 is below the noise floor.
> **Hybrid solo (v006) is the Phase 3 ceiling.**


In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

# Path detection — works in VSCode (walk-up) and on Kaggle (fixed path)
if Path('/kaggle/input').exists():
    parquets = list(Path('/kaggle/input').rglob('train_features_v2.parquet'))
    project_root = parquets[0].parents[2] if parquets else Path('/kaggle/working')
    processed_dir  = project_root / 'data' / 'processed'
    submissions_dir = Path('/kaggle/working')
    models_dir     = Path('/kaggle/working')
else:
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    project_root   = cwd
    processed_dir  = project_root / 'data' / 'processed'
    submissions_dir = project_root / 'submissions'
    models_dir     = project_root / 'models'

print(f'Project root  : {project_root}')
print(f'Processed dir : {processed_dir.exists()}')
print(f'Submissions   : {submissions_dir.exists()}')

BOOST = 0.0177  # confirmed neural boost (v006 Hybrid solo: CV 0.9185 → LB 0.93618)


Project root  : c:\Repos\predict-f1-pit-stops
Processed dir : True
Submissions   : True


## 1 — Load OOF Predictions


In [2]:
# Load all four OOF parquets
oof12 = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
oof15 = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')
oof18 = pd.read_parquet(processed_dir / 'oof_predictions_nb18.parquet')
oof21 = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')

# Verify row counts match
for name, df in [('NB12', oof12), ('NB15', oof15), ('NB18', oof18), ('NB21', oof21)]:
    print(f'{name}: {df.shape[0]} rows | cols: {df.columns.tolist()}')

assert all(len(df) == len(oof12) for df in [oof15, oof18, oof21]), 'Row count mismatch!'
print(f'\n✓ All OOF parquets: {len(oof12)} rows')


NB12: 439140 rows | cols: ['id', 'fold', 'PitNextLap', 'lgbm_tuned_oof', 'cb_tuned_oof']
NB15: 439140 rows | cols: ['id', 'fold', 'PitNextLap', 'mlp_oof']
NB18: 439140 rows | cols: ['id', 'fold', 'PitNextLap', 'lstm_oof']
NB21: 439140 rows | cols: ['id', 'fold', 'PitNextLap', 'hybrid_oof']

✓ All OOF parquets: 439140 rows


In [3]:
# Verify individual solo AUCs match expected values
y_true = oof12['PitNextLap'].values

auc_lgbm   = roc_auc_score(y_true, oof12['lgbm_tuned_oof'])
auc_mlp    = roc_auc_score(oof15['PitNextLap'], oof15['mlp_oof'])
auc_lstm   = roc_auc_score(oof18['PitNextLap'], oof18['lstm_oof'])
auc_hybrid = roc_auc_score(oof21['PitNextLap'], oof21['hybrid_oof'])

print('Solo AUC verification (expected → actual):')
print(f'  LGBM  NB12 : 0.9024 → {auc_lgbm:.4f}  {"✓" if abs(auc_lgbm - 0.9024) < 1e-3 else "MISMATCH"}')
print(f'  MLP   NB15 : 0.9102 → {auc_mlp:.4f}  {"✓" if abs(auc_mlp - 0.9102) < 1e-3 else "MISMATCH"}')
print(f'  LSTM  NB18 : 0.9122 → {auc_lstm:.4f}  {"✓" if abs(auc_lstm - 0.9122) < 1e-3 else "MISMATCH"}')
print(f'  Hybrid NB21: 0.9185 → {auc_hybrid:.4f}  {"✓" if abs(auc_hybrid - 0.9185) < 1e-3 else "MISMATCH"}')

print('\nRevised gate (solo AUC ≥ 0.905 AND ρ vs LGBM < 0.85):')
for name, auc in [('MLP', auc_mlp), ('LSTM', auc_lstm), ('Hybrid', auc_hybrid)]:
    gate_auc = auc >= 0.905
    print(f'  {name:<8}: AUC gate {"PASS ✓" if gate_auc else "FAIL ✗"} ({auc:.4f} {"≥" if gate_auc else "<"} 0.905) — ρ gate checked below')


Solo AUC verification (expected → actual):
  LGBM  NB12 : 0.9024 → 0.9024  ✓
  MLP   NB15 : 0.9102 → 0.9102  ✓
  LSTM  NB18 : 0.9122 → 0.9127  ✓
  Hybrid NB21: 0.9185 → 0.9185  ✓

Revised gate (solo AUC ≥ 0.905 AND ρ vs LGBM < 0.85):
  MLP     : AUC gate PASS ✓ (0.9102 ≥ 0.905) — ρ gate checked below
  LSTM    : AUC gate PASS ✓ (0.9127 ≥ 0.905) — ρ gate checked below
  Hybrid  : AUC gate PASS ✓ (0.9185 ≥ 0.905) — ρ gate checked below


## 2 — Full Spearman ρ Matrix


In [4]:
# Merge all OOF predictions on id
oof = oof12[['id', 'fold', 'PitNextLap', 'lgbm_tuned_oof']].copy()
oof = oof.merge(oof15[['id', 'mlp_oof']],    on='id')
oof = oof.merge(oof18[['id', 'lstm_oof']],   on='id')
oof = oof.merge(oof21[['id', 'hybrid_oof']], on='id')

assert oof.shape[0] == 439140, f'Expected 439140 rows, got {oof.shape[0]}'
assert oof['id'].duplicated().sum() == 0, 'Duplicate IDs!'
print(f'Merged OOF shape: {oof.shape}  ✓')

# Full 4×4 Spearman ρ matrix
model_cols = ['lgbm_tuned_oof', 'mlp_oof', 'lstm_oof', 'hybrid_oof']
model_names = ['LGBM', 'MLP', 'LSTM', 'Hybrid']

rho = {}
print('\nSpearman ρ matrix:')
print(f'{"":>10}', end='')
for n in model_names:
    print(f'{n:>10}', end='')
print()
for i, (n1, c1) in enumerate(zip(model_names, model_cols)):
    print(f'{n1:>10}', end='')
    for j, (n2, c2) in enumerate(zip(model_names, model_cols)):
        if i == j:
            print(f'{"—":>10}', end='')
        else:
            r, _ = spearmanr(oof[c1], oof[c2])
            rho[f'{n1}_{n2}'] = r
            flag = ' ✓' if r < 0.85 else ''
            print(f'{r:>9.3f}{flag}', end='')
    print()

print()
print('Gate (ρ < 0.85 vs LGBM):')
for name, col in [('MLP', 'mlp_oof'), ('LSTM', 'lstm_oof'), ('Hybrid', 'hybrid_oof')]:
    r = rho[f'LGBM_{name}']
    status = 'PASS ✓' if r < 0.85 else 'FAIL ✗'
    print(f'  {name:<8}: ρ vs LGBM = {r:.4f}  → {status}')


Merged OOF shape: (439140, 7)  ✓

Spearman ρ matrix:
                LGBM       MLP      LSTM    Hybrid
      LGBM         —    0.791 ✓    0.804 ✓    0.802 ✓
       MLP    0.791 ✓         —    0.915    0.919
      LSTM    0.804 ✓    0.915         —    0.911
    Hybrid    0.802 ✓    0.919    0.911         —

Gate (ρ < 0.85 vs LGBM):
  MLP     : ρ vs LGBM = 0.7910  → PASS ✓
  LSTM    : ρ vs LGBM = 0.8044  → PASS ✓
  Hybrid  : ρ vs LGBM = 0.8023  → PASS ✓


## 3 — Ensemble Comparison


In [5]:
def rank_average(arrays):
    """Equal-weight rank average. Returns normalized ranks in (0, 1)."""
    n = len(arrays[0])
    ranks = np.stack([rankdata(a) for a in arrays])
    return ranks.mean(axis=0) / (n + 1)


combos = [
    ('LGBM solo',                ['lgbm_tuned_oof'],                                    'NB12 reference'),
    ('MLP solo',                 ['mlp_oof'],                                            'NB15 reference'),
    ('LSTM solo',                ['lstm_oof'],                                           'NB18 reference'),
    ('Hybrid solo',              ['hybrid_oof'],                                         'NB21 reference'),
    ('LGBM + MLP',               ['lgbm_tuned_oof', 'mlp_oof'],                         'v004 baseline ★'),
    ('LGBM + MLP + LSTM',        ['lgbm_tuned_oof', 'mlp_oof', 'lstm_oof'],             '3-model'),
    ('LGBM + MLP + Hybrid',      ['lgbm_tuned_oof', 'mlp_oof', 'hybrid_oof'],           '3-model'),
    ('LGBM + MLP + LSTM + Hybrid',['lgbm_tuned_oof','mlp_oof','lstm_oof','hybrid_oof'], '4-model CHOSEN ★★'),
]

print(f'{"Combination":<32} {"CV AUC":>8}  {"Δ v004":>8}  {"Proj LB":>9}  Note')
print('-' * 80)

results = {}
v004_auc = None
for name, cols, note in combos:
    preds = [oof[c].values for c in cols]
    avg = rank_average(preds) if len(preds) > 1 else preds[0]
    auc = roc_auc_score(oof['PitNextLap'], avg)
    results[name] = {'auc': auc, 'cols': cols, 'preds': avg}
    if name == 'LGBM + MLP':
        v004_auc = auc
    delta = auc - v004_auc if v004_auc else 0.0
    proj = auc + BOOST
    print(f'  {name:<30} {auc:.4f}   {delta:+.4f}   {proj:.4f}   {note}')

chosen_name = 'LGBM + MLP + LSTM + Hybrid'
chosen_auc  = results[chosen_name]['auc']
print(f'\nChosen: {chosen_name}')
print(f'  CV AUC   : {chosen_auc:.4f}')
print(f'  Proj LB  : ~{chosen_auc + BOOST:.4f}  (using +{BOOST:.4f} confirmed boost)')
print(f'  Δ v004   : {chosen_auc - v004_auc:+.4f} CV  →  ~{(chosen_auc + BOOST) - 0.93160:+.4f} LB (est.)')


Combination                        CV AUC    Δ v004    Proj LB  Note
--------------------------------------------------------------------------------
  LGBM solo                      0.9024   +0.0000   0.9201   NB12 reference
  MLP solo                       0.9102   +0.0000   0.9279   NB15 reference
  LSTM solo                      0.9127   +0.0000   0.9304   NB18 reference
  Hybrid solo                    0.9185   +0.0000   0.9362   NB21 reference
  LGBM + MLP                     0.9150   +0.0000   0.9327   v004 baseline ★
  LGBM + MLP + LSTM              0.9179   +0.0029   0.9356   3-model
  LGBM + MLP + Hybrid            0.9196   +0.0047   0.9373   3-model
  LGBM + MLP + LSTM + Hybrid     0.9205   +0.0056   0.9382   4-model CHOSEN ★★

Chosen: LGBM + MLP + LSTM + Hybrid
  CV AUC   : 0.9205
  Proj LB  : ~0.9382  (using +0.0177 confirmed boost)
  Δ v004   : +0.0056 CV  →  ~+0.0066 LB (est.)


## 4 — Test Predictions


In [6]:
# Load all four test prediction parquets
test12 = pd.read_parquet(processed_dir / 'test_predictions_nb12.parquet')
test15 = pd.read_parquet(processed_dir / 'test_predictions_nb15.parquet')
test18 = pd.read_parquet(processed_dir / 'test_predictions_nb18.parquet')
test21 = pd.read_parquet(processed_dir / 'test_predictions_nb21.parquet')

for name, df, col in [('NB12', test12, 'lgbm_tuned_pred'), ('NB15', test15, 'mlp_pred'),
                       ('NB18', test18, 'lstm_pred'),       ('NB21', test21, 'hybrid_pred')]:
    print(f'{name}: {df.shape[0]} rows | col {col!r} present: {col in df.columns}')

# Merge on id
test = test12[['id', 'lgbm_tuned_pred']].copy()
test = test.merge(test15[['id', 'mlp_pred']],    on='id')
test = test.merge(test18[['id', 'lstm_pred']],   on='id')
test = test.merge(test21[['id', 'hybrid_pred']], on='id')

assert test.shape[0] == 188165, f'Expected 188165 rows, got {test.shape[0]}'
assert test['id'].duplicated().sum() == 0, 'Duplicate IDs in test!'
print(f'\nMerged test shape: {test.shape}  ✓')
print(f'ID range: {test["id"].min()} – {test["id"].max()}')


NB12: 188165 rows | col 'lgbm_tuned_pred' present: True
NB15: 188165 rows | col 'mlp_pred' present: True
NB18: 188165 rows | col 'lstm_pred' present: True
NB21: 188165 rows | col 'hybrid_pred' present: True

Merged test shape: (188165, 5)  ✓
ID range: 439140 – 627304


## 5 — Build Submission


In [7]:
# Rank average: LGBM + MLP + LSTM + Hybrid (equal weight)
test['PitNextLap'] = rank_average([
    test['lgbm_tuned_pred'].values,
    test['mlp_pred'].values,
    test['lstm_pred'].values,
    test['hybrid_pred'].values,
])

submission = test[['id', 'PitNextLap']].sort_values('id').reset_index(drop=True)

# Sanity checks
assert submission['id'].duplicated().sum() == 0,       'Duplicate IDs!'
assert submission['PitNextLap'].isnull().sum() == 0,   'NaN predictions!'
assert submission['PitNextLap'].between(0, 1).all(),   'Predictions out of [0,1]!'

print('Submission shape:', submission.shape)
print(submission['PitNextLap'].describe().round(6).to_string())
print('\nFirst 5 rows:')
print(submission.head().to_string(index=False))

submission_path = submissions_dir / 'submission_v007_super_ensemble.csv'
submission.to_csv(submission_path, index=False)
print(f'\n✓ Saved: {submission_path}')
print(f'  File size: {submission_path.stat().st_size / 1e6:.2f} MB')


Submission shape: (188165, 2)
count    188165.000000
mean          0.500000
std           0.280711
min           0.000185
25%           0.257888
50%           0.482511
75%           0.749435
max           0.999932

First 5 rows:
    id  PitNextLap
439140    0.207003
439141    0.294487
439142    0.369505
439143    0.705440
439144    0.878468

✓ Saved: c:\Repos\predict-f1-pit-stops\submissions\submission_v007_super_ensemble.csv
  File size: 5.13 MB


In [8]:
# Save summary pickle
summary = {
    'ensemble_strategy'  : 'rank_avg_4model',
    'models_included'    : ['LGBM_NB12', 'MLP_NB15', 'LSTM_NB18', 'Hybrid_NB21'],
    'models_excluded'    : ['TFT_NB19'],
    'exclusion_reason'   : {'TFT_NB19': 'AUC gate FAIL — 0.8483 < 0.905'},
    'solo_aucs'          : {
        'lgbm': auc_lgbm, 'mlp': auc_mlp, 'lstm': auc_lstm, 'hybrid': auc_hybrid,
    },
    'spearman_rho'       : rho,
    'ensemble_results'   : {k: v['auc'] for k, v in results.items()},
    'chosen_combo'       : chosen_name,
    'chosen_cv_auc'      : chosen_auc,
    'boost_used'         : BOOST,
    'projected_lb'       : round(chosen_auc + BOOST, 4),
    'submission_file'    : submission_path.name,
    'submission_rows'    : len(submission),
}
with open(models_dir / 'nb20_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)
print('✓ Saved models/nb20_summary.pkl')

print()
print('=' * 65)
print('NB20 — Super-Ensemble Summary')
print('=' * 65)
print(f'  Models          : LGBM NB12 + MLP NB15 + LSTM NB18 + Hybrid NB21')
print(f'  Ensemble CV AUC : {chosen_auc:.4f}')
print(f'  Δ vs v004       : {chosen_auc - v004_auc:+.4f}  (v004 = 0.9150)')
print(f'  Boost used      : +{BOOST:.4f}  (confirmed v006 Hybrid solo)')
print(f'  Projected LB    : ~{chosen_auc + BOOST:.4f}')
print(f'  Best LB so far  : 0.93618  (v006 Hybrid solo)')
print(f'  Expected gain   : ~{(chosen_auc + BOOST) - 0.93618:+.4f} LB points')
print(f'  LB top          : 0.95488')
print(f'  Remaining gap   : ~{0.95488 - (chosen_auc + BOOST):.4f}')
print(f'  Submission file : {submission_path.name}')
print()
print('Add to leaderboard_log.md after uploading:')
print(f'| {submission_path.name} | rank avg LGBM+MLP+LSTM+Hybrid | {chosen_auc:.4f} | [LB AUC] | Submitted 2026-05-23 |')


✓ Saved models/nb20_summary.pkl

NB20 — Super-Ensemble Summary
  Models          : LGBM NB12 + MLP NB15 + LSTM NB18 + Hybrid NB21
  Ensemble CV AUC : 0.9205
  Δ vs v004       : +0.0056  (v004 = 0.9150)
  Boost used      : +0.0177  (confirmed v006 Hybrid solo)
  Projected LB    : ~0.9382
  Best LB so far  : 0.93618  (v006 Hybrid solo)
  Expected gain   : ~+0.0021 LB points
  LB top          : 0.95488
  Remaining gap   : ~0.0166
  Submission file : submission_v007_super_ensemble.csv

Add to leaderboard_log.md after uploading:
| submission_v007_super_ensemble.csv | rank avg LGBM+MLP+LSTM+Hybrid | 0.9205 | [LB AUC] | Submitted 2026-05-23 |


## NB20 Result

**Submission:** `submission_v007_super_ensemble.csv`
**LB AUC: 0.93463** — boost +0.0141 (projected +0.0177, delta −0.0036)

| Metric | Value |
|--------|-------|
| Ensemble CV AUC | 0.9205 |
| Actual LB AUC | **0.93463** |
| Actual boost | +0.0141 |
| vs Hybrid solo (0.93618) | **−0.00155** |
| vs v004 baseline (0.93160) | +0.00303 |

### Why the ensemble hurt vs Hybrid solo

| Combo | CV AUC | Proj LB | Actual LB |
|-------|--------|---------|-----------|
| Hybrid solo (v006) | 0.9185 | 0.9362 | **0.93618** ← best |
| MLP + Hybrid | 0.9185 | 0.9362 | — (zero CV gain; ρ=0.9186) |
| LGBM + Hybrid | 0.9191 | 0.9368 | — (+0.0006 CV, below noise floor) |
| 4-model (this NB) | 0.9205 | 0.9382 | **0.93463** ← submitted |

MLP is a near-duplicate of Hybrid on this dataset (ρ=0.9186 — adding MLP to Hybrid rank average
produces the same CV as Hybrid alone). LSTM is similarly correlated (ρ=0.9111). Both dilute
Hybrid's strong test-set signal without adding complementary errors. The only genuinely diverse
partner is LGBM (ρ=0.8023), but its CV contribution (+0.0006) falls below the noise floor.

**Lesson:** When the best model has ρ ~0.91 vs all neural partners, no neural ensemble beats the
solo model on LB. The CV gain from including highly-correlated models reflects GroupKFold CV
structure, not generalizable signal. **Phase 3 complete — Hybrid solo (0.93618) is final best.**
